In [1]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt



def charger_grille(chemin_image, n_cols, n_rows, taille_pixel=28):
    """
    Découpe une image grille en chiffres individuels 28×28.
    Retourne un tableau numpy (N, 784).
    """
    img = Image.open(chemin_image).convert('L')  # niveaux de gris
    img_array = np.array(img)

    H, W = img_array.shape
    h_chiffre = H // n_rows
    w_chiffre = W // n_cols

    images = []
    for row in range(n_rows):
        for col in range(n_cols):
            y0 = row * h_chiffre
            y1 = y0 + h_chiffre
            x0 = col * w_chiffre
            x1 = x0 + w_chiffre

            chiffre = img_array[y0:y1, x0:x1]
            # Redimensionner à 28×28 si nécessaire
            chiffre_pil = Image.fromarray(chiffre).resize((28, 28))
            chiffre_vec = np.array(chiffre_pil).flatten()  # vecteur ℝ^784
            images.append(chiffre_vec)

    return np.array(images)

In [2]:
from tensorflow.keras.datasets import mnist  # ou : sklearn, torchvision

(X_train, y_train), (X_test, y_test) = mnist.load_data()

# X_train : (60000, 28, 28) → on vectorise en (60000, 784)
X_train = X_train.reshape(60000, 784)
X_test = X_test.reshape(10000, 784)

# ── 3. Normalisation ─────────────────────────────────────────────────────────
# Pixels entre 0 et 255 → on ramène entre 0 et 1
X_train = X_train / 255.0
X_test = X_test / 255.0

print(f"X_train : {X_train.shape}, valeurs : [{X_train.min():.2f}, {X_train.max():.2f}]")
print(f"X_test  : {X_test.shape}")
print(f"Classes présentes : {np.unique(y_train)}")

X_train : (60000, 784), valeurs : [0.00, 1.00]
X_test  : (10000, 784)
Classes présentes : [0 1 2 3 4 5 6 7 8 9]


In [3]:
def afficher_grille(X, y, n=10):
    fig, axes = plt.subplots(1, n, figsize=(15, 2))
    for i, ax in enumerate(axes):
        ax.imshow(X[i].reshape(28, 28), cmap='gray')
        ax.set_title(f"y={y[i]}")
        ax.axis('off')
    plt.tight_layout()
    plt.show()

In [4]:
import numpy as np

# ── Initialisation ─────────────────────────────
n_features = 784
n_classes = 10

W = np.random.randn(n_classes, n_features) * 0.01
b = np.zeros((n_classes, 1))

# ── Softmax ───────────────────────────────────
def softmax(z):
    z = z - np.max(z, axis=0, keepdims=True)  # stabilité numérique
    exp_z = np.exp(z)
    return exp_z / np.sum(exp_z, axis=0, keepdims=True)

# ── One-hot ───────────────────────────────────
def one_hot(y, num_classes=10):
    oh = np.zeros((num_classes, y.size))
    oh[y, np.arange(y.size)] = 1
    return oh

# ── Entraînement ─────────────────────────────
def train(X, y, lr=0.1, epochs=10):
    global W, b

    X = X.T  # shape (784, N)
    y_onehot = one_hot(y)

    for epoch in range(epochs):

        # Forward
        z = W @ X + b
        y_hat = softmax(z)

        # Loss
        loss = -np.mean(np.sum(y_onehot * np.log(y_hat + 1e-9), axis=0))

        # Gradient
        dz = y_hat - y_onehot
        dW = (dz @ X.T) / X.shape[1]
        db = np.mean(dz, axis=1, keepdims=True)

        # Update
        W -= lr * dW
        b -= lr * db

        print(f"Epoch {epoch+1}, Loss = {loss:.4f}")

# ── Prédiction ────────────────────────────────
def predict(X):
    X = X.T
    z = W @ X + b
    y_hat = softmax(z)
    return np.argmax(y_hat, axis=0)

In [5]:
train(X_train, y_train, lr=0.1, epochs=20)

y_pred = predict(X_test)

accuracy = np.mean(y_pred == y_test)
print("Accuracy :", accuracy)

Epoch 1, Loss = 2.2954
Epoch 2, Loss = 2.1900
Epoch 3, Loss = 2.0939
Epoch 4, Loss = 2.0052
Epoch 5, Loss = 1.9230
Epoch 6, Loss = 1.8468
Epoch 7, Loss = 1.7761
Epoch 8, Loss = 1.7106
Epoch 9, Loss = 1.6499
Epoch 10, Loss = 1.5936
Epoch 11, Loss = 1.5414
Epoch 12, Loss = 1.4930
Epoch 13, Loss = 1.4480
Epoch 14, Loss = 1.4062
Epoch 15, Loss = 1.3673
Epoch 16, Loss = 1.3311
Epoch 17, Loss = 1.2972
Epoch 18, Loss = 1.2656
Epoch 19, Loss = 1.2360
Epoch 20, Loss = 1.2083
Accuracy : 0.8193


In [6]:
import torch
import torch . nn as nn
import numpy as np
import matplotlib . pyplot as plt

import numpy as np

# ── Activation ───────────────────────────────
def relu(x):
    return np.maximum(0, x)

def relu_deriv(x):
    return (x > 0).astype(float)

def softmax(z):
    z = z - np.max(z, axis=0, keepdims=True)
    exp_z = np.exp(z)
    return exp_z / np.sum(exp_z, axis=0, keepdims=True)

# ── One-hot ─────────────────────────────────
def one_hot(y, num_classes=10):
    oh = np.zeros((num_classes, y.size))
    oh[y, np.arange(y.size)] = 1
    return oh

# ── Initialisation ──────────────────────────
def init_params(layer_sizes):
    params = {}
    for l in range(1, len(layer_sizes)):
        params["W"+str(l)] = np.random.randn(layer_sizes[l], layer_sizes[l-1]) * 0.01
        params["b"+str(l)] = np.zeros((layer_sizes[l], 1))
    return params

# ── Forward pass ────────────────────────────
def forward(X, params):
    cache = {}
    A = X.T  # (784, N)

    cache["A0"] = A

    L = len(params) // 2

    for l in range(1, L):
        Z = params["W"+str(l)] @ A + params["b"+str(l)]
        A = relu(Z)
        cache["Z"+str(l)] = Z
        cache["A"+str(l)] = A

    # Dernière couche (scores)
    ZL = params["W"+str(L)] @ A + params["b"+str(L)]
    AL = softmax(ZL)

    cache["Z"+str(L)] = ZL
    cache["A"+str(L)] = AL

    return AL, cache

# ── Loss ────────────────────────────────────
def compute_loss(AL, Y):
    m = Y.shape[1]
    return -np.sum(Y * np.log(AL + 1e-9)) / m

# ── Backprop ────────────────────────────────
def backward(X, Y, params, cache):
    grads = {}
    m = X.shape[0]
    L = len(params) // 2

    # Sortie
    dZ = cache["A"+str(L)] - Y

    for l in reversed(range(1, L+1)):
        A_prev = cache["A"+str(l-1)]

        grads["dW"+str(l)] = (dZ @ A_prev.T) / m
        grads["db"+str(l)] = np.sum(dZ, axis=1, keepdims=True) / m

        if l > 1:
            dA_prev = params["W"+str(l)].T @ dZ
            dZ = dA_prev * relu_deriv(cache["Z"+str(l-1)])

    return grads

# ── Update ─────────────────────────────────
def update(params, grads, lr):
    L = len(params) // 2
    for l in range(1, L+1):
        params["W"+str(l)] -= lr * grads["dW"+str(l)]
        params["b"+str(l)] -= lr * grads["db"+str(l)]
    return params

# ── Training ───────────────────────────────
def train(X, y, layer_sizes, lr=0.1, epochs=10):
    params = init_params(layer_sizes)
    Y = one_hot(y)

    for epoch in range(epochs):
        AL, cache = forward(X, params)
        loss = compute_loss(AL, Y)
        grads = backward(X, Y, params, cache)
        params = update(params, grads, lr)

        print(f"Epoch {epoch+1}, Loss = {loss:.4f}")

    return params

# ── Prediction ─────────────────────────────
def predict(X, params):
    AL, _ = forward(X, params)
    return np.argmax(AL, axis=0)

In [7]:
layer_sizes = [784, 128, 64, 10]  # 2 couches cachées

params = train(X_train, y_train, layer_sizes, lr=0.1, epochs=20)

y_pred = predict(X_test, params)

accuracy = np.mean(y_pred == y_test)
print("Accuracy :", accuracy)

Epoch 1, Loss = 2.3025
Epoch 2, Loss = 2.3025
Epoch 3, Loss = 2.3025
Epoch 4, Loss = 2.3024
Epoch 5, Loss = 2.3024
Epoch 6, Loss = 2.3024
Epoch 7, Loss = 2.3023
Epoch 8, Loss = 2.3023
Epoch 9, Loss = 2.3023
Epoch 10, Loss = 2.3022
Epoch 11, Loss = 2.3022
Epoch 12, Loss = 2.3022
Epoch 13, Loss = 2.3022
Epoch 14, Loss = 2.3021
Epoch 15, Loss = 2.3021
Epoch 16, Loss = 2.3021
Epoch 17, Loss = 2.3020
Epoch 18, Loss = 2.3020
Epoch 19, Loss = 2.3020
Epoch 20, Loss = 2.3020
Accuracy : 0.1135
